This notebook turns my YOLO-format labels into Pascal VOC-style XML
annotations, because the Faster R-CNN code expects VOC XML files instead
of YOLO's plain text boxes. There are two things happening below: the
actual conversion, and a quick visual check afterward so I can see the
converted boxes drawn on an image and confirm nothing got mixed up.

**Step 1: convert.** Reads every YOLO `.txt` label in `new_dataset/labels/val`,
works out the matching image so it knows the image size (needed because YOLO
boxes are normalized 0-1, VOC boxes are pixel coordinates), and writes out a
VOC XML file per image into `new_dataset/labels_voc/val`.

In [ ]:

import os
import xml.etree.ElementTree as ET
from xml.dom.minidom import parseString
from PIL import Image

def yolo_to_voc_coords(x_center, y_center, width, height, img_width, img_height):
    x_center *= img_width
    y_center *= img_height
    width *= img_width
    height *= img_height
    x_min = int(x_center - width / 2)
    y_min = int(y_center - height / 2)
    x_max = int(x_center + width / 2)
    y_max = int(y_center + height / 2)
    return x_min, y_min, x_max, y_max

def create_voc_xml(filename, size, objects):
    annotation = ET.Element("annotation")

    ET.SubElement(annotation, "folder").text = "images"
    ET.SubElement(annotation, "filename").text = filename
    ET.SubElement(annotation, "path").text = filename

    source = ET.SubElement(annotation, "source")
    ET.SubElement(source, "database").text = "Unknown"

    size_elem = ET.SubElement(annotation, "size")
    ET.SubElement(size_elem, "width").text = str(size[0])
    ET.SubElement(size_elem, "height").text = str(size[1])
    ET.SubElement(size_elem, "depth").text = str(size[2])

    ET.SubElement(annotation, "segmented").text = "0"

    for obj in objects:
        obj_elem = ET.SubElement(annotation, "object")
        ET.SubElement(obj_elem, "name").text = obj['name']
        ET.SubElement(obj_elem, "pose").text = "Unspecified"
        ET.SubElement(obj_elem, "truncated").text = "0"
        ET.SubElement(obj_elem, "difficult").text = "0"

        bbox = ET.SubElement(obj_elem, "bndbox")
        ET.SubElement(bbox, "xmin").text = str(obj['bbox'][0])
        ET.SubElement(bbox, "ymin").text = str(obj['bbox'][1])
        ET.SubElement(bbox, "xmax").text = str(obj['bbox'][2])
        ET.SubElement(bbox, "ymax").text = str(obj['bbox'][3])

    return parseString(ET.tostring(annotation)).toprettyxml(indent="  ")

def convert_yolo_to_voc(yolo_dir, img_dir, voc_dir, class_names):
    os.makedirs(voc_dir, exist_ok=True)

    for file in os.listdir(yolo_dir):
        if not file.endswith(".txt"):
            continue

        image_file = file.replace(".txt", ".jpg")
        image_path = os.path.join(img_dir, image_file)
        if not os.path.exists(image_path):
            image_file = file.replace(".txt", ".png")
            image_path = os.path.join(img_dir, image_file)
            if not os.path.exists(image_path):
                print(f"Image for {file} not found.")
                continue

        with Image.open(image_path) as img:
            img_width, img_height = img.size
            img_depth = len(img.getbands())

        yolo_path = os.path.join(yolo_dir, file)
        objects = []

        with open(yolo_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls_id, x, y, w, h = map(float, parts)
                x_min, y_min, x_max, y_max = yolo_to_voc_coords(x, y, w, h, img_width, img_height)
                cls_name = class_names[int(cls_id)]
                objects.append({"name": cls_name, "bbox": (x_min, y_min, x_max, y_max)})

        voc_xml = create_voc_xml(image_file, (img_width, img_height, img_depth), objects)
        voc_path = os.path.join(voc_dir, file.replace(".txt", ".xml"))

        with open(voc_path, "w") as f:
            f.write(voc_xml)

        print(f"Converted {file} to {voc_path}")

if __name__ == "__main__":
    yolo_label_dir = "new_dataset/labels/val"
    image_dir = "new_dataset/images/val"
    voc_output_dir = "new_dataset/labels_voc/val"
    class_names = ["fall", "no_fall"]

    convert_yolo_to_voc(yolo_label_dir, image_dir, voc_output_dir, class_names)

**Step 2: sanity check.** Just picks one converted image + XML pair and
draws the boxes on top of it, so I can eyeball whether the conversion
actually lined up correctly before trusting it for the rest of the dataset.

In [ ]:
import os
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def class_to_idx(class_name):
    class_map = {
        "fall": 0,
        "no_fall": 1,
    }
    return class_map.get(class_name, -1)

def visualize_annotation(image_path, xml_path):
    image = Image.open(image_path).convert("RGB")
    tree = ET.parse(xml_path)
    root = tree.getroot()

    fig, ax = plt.subplots(1)
    ax.imshow(image)

    for obj in root.findall("object"):
        class_name = obj.find("name").text
        bbox = obj.find("bndbox")
        xmin = int(float(bbox.find("xmin").text))
        ymin = int(float(bbox.find("ymin").text))
        xmax = int(float(bbox.find("xmax").text))
        ymax = int(float(bbox.find("ymax").text))

        rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                 linewidth=2, edgecolor='r', facecolor='none')
        ax.add_patch(rect)
        ax.text(xmin, ymin - 5, f"{class_name} ({class_to_idx(class_name)})",
                color='red', fontsize=10, backgroundcolor="white")

    plt.axis("off")
    plt.show()

# Example usage
img_path = "dataset_paper_new/images/train/no_fall_ (200).png"
xml_path = "dataset_paper_new/labels_voc/train/no_fall_ (200).xml"
visualize_annotation(img_path, xml_path)
